Problema Monty Hall:

Avem trei usi, in spatele uneia este premiul, in spatele celorlalte nu. Jucatorul alege initial o usa, prezentatorul stie unde este premiul si deschide una dintre celelalte usi unde stie ca nu este premiul. Dupa, jucatorul poate alege daca sa ramana sau nu la alegerea facuta anterioara. Intrebarea este daca este mai avantajos sa te schimbi.
Vom arata ca este.

Luam trei variabile:
C = usa aleasa de concurent
P = usa unde e premiul
H = usa deschisa de prezentator

Usile vor fi 0,1,2
Structura retelei: C->H<-P pentru ca usa prezentatorului depinde atat de premiu cat si de alegerea noastra.

Deci dupa formula: P(C,P,H) = P(C)P(P)P(H|C,P)

pt ca C si P sunt independente.

P(C=1)=P(C=2)=P(C=3) = 1/3


Premiul la fel e aleator
P(P=1)=P(P=2)=P(P=3) = 1/3



In [2]:
from pgmpy.factors.discrete import TabularCPD

cpd_c = TabularCPD(variable ="C",variable_card = 3, values=[[1/3],[1/3],[1/3]])
print (cpd_c)

cpd_p = TabularCPD(variable ="P",variable_card = 3, values=[[1/3],[1/3],[1/3]])
print (cpd_p)



+------+----------+
| C(0) | 0.333333 |
+------+----------+
| C(1) | 0.333333 |
+------+----------+
| C(2) | 0.333333 |
+------+----------+
+------+----------+
| P(0) | 0.333333 |
+------+----------+
| P(1) | 0.333333 |
+------+----------+
| P(2) | 0.333333 |
+------+----------+


Pentru prezentator, situatia sta altfel:

El respecta 2 conditii: nu poate alege usa aleasa de concurent, nu poate deschide usa cu premiul
Daca concurentul alege premiul: C=0,P=0 atunci prezentatorul poate alege intre 1 si 2 P(H=1|C=0,P=0) = P(H=2|C=0,P=0) = 1/2
Iar daca concurentul nu alege premiul, prezentatorul nu mai are libertate: C=0,P=1 atunci el nu poate alege 0 ca acolo e concurentul, nu poate alege 1 ca acolo e premiul, deci ii ramane doar P(H=2| C=0,P=1) = 1.


In [3]:
cpd_h = TabularCPD(variable = "H", variable_card=3,
    values = [
        [0,0,0,0,0.5,1,0,1,0.5],
        [0.5,0,1,0,0,0,1,0,0.5],
        [0.5,1,0,1,0.5,0,0,0,0]
    ],

    evidence = ["C","P"],
    evidence_card=[3,3]
)
print(cpd_h)

+------+------+------+------+------+------+------+------+------+------+
| C    | C(0) | C(0) | C(0) | C(1) | C(1) | C(1) | C(2) | C(2) | C(2) |
+------+------+------+------+------+------+------+------+------+------+
| P    | P(0) | P(1) | P(2) | P(0) | P(1) | P(2) | P(0) | P(1) | P(2) |
+------+------+------+------+------+------+------+------+------+------+
| H(0) | 0.0  | 0.0  | 0.0  | 0.0  | 0.5  | 1.0  | 0.0  | 1.0  | 0.5  |
+------+------+------+------+------+------+------+------+------+------+
| H(1) | 0.5  | 0.0  | 1.0  | 0.0  | 0.0  | 0.0  | 1.0  | 0.0  | 0.5  |
+------+------+------+------+------+------+------+------+------+------+
| H(2) | 0.5  | 1.0  | 0.0  | 1.0  | 0.5  | 0.0  | 0.0  | 0.0  | 0.0  |
+------+------+------+------+------+------+------+------+------+------+


Coloanele corespund combinatiilor C si P, {0,1,2}^2, iar randurile H  : 0,1,2

Vom adauga totul la model, apoi presupunem ca C = 0, H=2, vrem sa aflam care e probabilitatea : P(P|C=0,H=2)

In [4]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination
model = DiscreteBayesianNetwork([("C","H"),("P","H")])
model.add_cpds(cpd_c,cpd_p,cpd_h)
print(model.check_model())

infer = VariableElimination(model)
posterior_p = infer.query(variables=["P"],evidence={"C":0,"H":2})
print(posterior_p)

C:\Users\Badan Adrian\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Badan Adrian\AppData\Roaming\Python\Python314\site-packages\pgmpy\estimators\__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


True
+------+----------+
| P    |   phi(P) |
+======+==========+
| P(0) |   0.3333 |
+------+----------+
| P(1) |   0.6667 |
+------+----------+
| P(2) |   0.0000 |
+------+----------+


Valorile pot fi verificate si manual, prin formula prezentata la inceput. Respectiv usa ramasa, are probabilitate mai mare, deci schimbam.
